In [23]:
%useLatestDescriptors
%use dataframe
@file:DependsOn("com.github.doyaaaaaken:kotlin-csv-jvm:1.7.0")

import com.github.doyaaaaaken.kotlincsv.dsl.csvWriter
import io.github.oshai.kotlinlogging.KotlinLogging.logger
import kotlin.reflect.full.declaredMemberProperties
import java.nio.file.Paths
import java.util.Locale
import kotlin.io.path.Path

enum class Mode { FLAT, RANDOM }
enum class Algorithm(val shortName: String) {
    FROMBACK("tsprcs"),
    DISTANCE("tsprce"),
    SPARSITY("tsprcs"),
    OP("op"),
}
enum class Context { ELIMINATION, BUDGET, CLUSTERING }
enum class BudgetFactor(val string: String, val value: Int) {
    BUDGET_30("0.3", 30),
    BUDGET_50("0.5", 50),
    BUDGET_70("0.7", 70),
    BUDGET_100("1.0", 100)
}
enum class Parameter(val value: String) {
    CLUSTER_ELIMINATION_THRESHOLD("clustereliminationthreshold"),
    ALPHA("clustereliminationrevenueweight"),
    BETA("clustereliminationsparsityweight"),
    EPSILON("maxbudgetfactor"),
    ZETA("budgetweight")
}

enum class BudgetWeight(val value: String) {
    ELZEIN("el"),
    EQUAL("eq"),
    SPARSITY("cs"),
    DISTANCE("cd")
}

enum class BudgetMin(val value: String) {
    MIN("ml"),
    CENTER("c"),
    NONE("")
}

enum class UseMax(val value: String) {
    USEMAX("mx"),
    NOUSEMAX(""),
}

val percentageFraction = 1
val colsWithoutPercentages = "0.50"
val gradient = 0.1
val width = 1.9

val context = Context.BUDGET
val parameter = Parameter.EPSILON
val mode = Mode.FLAT
val algorithm = Algorithm.OP
val budgetFactor = BudgetFactor.BUDGET_100

val budgetWeight = BudgetWeight.ELZEIN
val budgetMin = BudgetMin.MIN
val useMax = UseMax.USEMAX


val fileName = when (parameter) {
    Parameter.ALPHA -> "param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmn_${algorithm.shortName}_50_${budgetWeight.value}${budgetMin.value}${useMax.value}_${parameter.value}_0.0_1.25_0.25.csv"
    Parameter.CLUSTER_ELIMINATION_THRESHOLD -> "param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmn_${algorithm.shortName}_50_${budgetWeight.value}${budgetMin.value}${useMax.value}_${parameter.value}_0.3_0.7_0.1.csv"
    Parameter.BETA -> "param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmn_${algorithm.shortName}_50_${budgetWeight.value}${budgetMin.value}${useMax.value}_${parameter.value}_-0.25_1.25_0.25.csv"
    Parameter.EPSILON -> "param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmd_${algorithm.shortName}_50_${budgetWeight.value}${budgetMin.value}${useMax.value}_${parameter.value}_0.1_0.7_0.1.csv"
    Parameter.ZETA -> "param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmd_${algorithm.shortName}_50_${budgetWeight.value}${budgetMin.value}${useMax.value}_${parameter.value}_1.0_7.0_1.0.csv"
}

val relativePath = "/op-solver-strict/results/${context.name.lowercase()}/"
val navigationPath = Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath().toString()

val path = Paths.get(navigationPath, relativePath, fileName).toString()

var df = DataFrame.readCsv(path)
df = df.remove { df.columns()[1] }
df

name,0.10,0.20,0.30,0.40,0.50,0.60,0.70
eil101,58.000000,59.000000,59.000000,60.000000,59.000000,59.000000,60.000000
gil262,132.000000,136.000000,140.000000,136.000000,134.000000,135.000000,139.000000
pr299,150.000000,147.000000,148.000000,130.000000,149.000000,152.000000,153.000000
lin318,179.000000,182.000000,185.000000,182.000000,177.000000,180.000000,185.000000
rd400,192.000000,200.000000,198.000000,195.000000,196.000000,195.000000,174.000000
d493,303.000000,302.000000,280.000000,276.000000,274.000000,288.000000,294.000000
u574,310.000000,308.000000,309.000000,307.000000,315.000000,305.000000,306.000000
u724,389.000000,388.000000,380.000000,383.000000,374.000000,387.000000,381.000000
pcb1173,578.000000,570.000000,575.000000,576.000000,567.000000,566.000000,564.000000
fl1400,983.000000,956.000000,877.000000,712.000000,663.000000,656.000000,681.000000


In [24]:
val bestValues = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        0
    } else {
        (col[row] as Double).toInt()
    }
}.map { row ->
    row.rowMaxOf<Int>()
}

bestValues

[60, 140, 153, 185, 200, 303, 315, 389, 578, 983, 1198]

In [25]:

 //max(maxRevenueDif,0.0)

val rowMaxValues = bestValues.mapIndexed { index, resultMax -> max(resultMax, (df[colsWithoutPercentages][index] as Number).toInt()) }

rowMaxValues

[60, 140, 153, 185, 200, 303, 315, 389, 578, 983, 1198]

In [26]:
val rowMinValues = df.map { row ->
    row.rowMinOfOrNull<Double>()
}.map { row -> row!!.toInt()}
rowMinValues

[58, 132, 130, 177, 174, 274, 305, 374, 564, 656, 1180]

In [27]:

val revenueDif = rowMaxValues.mapIndexed { index, maxEntry ->
    val minEntry = df[index].rowMinOf<Double>()
    (maxEntry - minEntry!!).toDouble() / maxEntry.toDouble()
}

fun getSaturation(gradient: Double, maxValue: Int, value: Int): String {
    return min(max((100 - ((maxValue - value) / (maxValue * gradient) * 100)), 0.0), 100.0).toInt().toString()
}

fun calculatePercentage(refValue: Int, compValue: Int): Double {
    return ((compValue.toDouble() - refValue.toDouble()) / refValue.toDouble())
}

fun formatePercentage(value: Double): String {
    return "${String.format(Locale.US, "%+.${percentageFraction}f", value * 100)}\\%"
}

fun getPercentage(row: DataRow<*>, compValue: Int): String {
    val refValue = (df.get(colsWithoutPercentages)[row] as Double).toInt()
    val percentage = calculatePercentage(refValue, compValue)
    return "{\\tiny${formatePercentage(percentage)}}"
}

val footer = df.convert { all() }.perRowCol { row, col ->
    if (col.name() == colsWithoutPercentages || col[row] is String) {
        10000.0
    } else {
        val refValue = (df.get(colsWithoutPercentages)[row] as Double).toInt()
        calculatePercentage(refValue, (col[row] as Double).toInt())
    }
}.mean().values().mapIndexed { index, it ->
    if (index == 0) {
        "avg diff"
    } else if (it is Double && it > 100.0) {
        "-"
    } else if (it is Double) {
        formatePercentage(it)
    } else {
        "${it.toString()}\\%"
    }
}.toList()
footer

[avg diff, +5.6\%, +5.7\%, +4.1\%, +0.2\%, -, +0.9\%, +1.0\%]

In [28]:

val stringdf = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        col[row].toString().split("-").first()
    } else if (col.name() == colsWithoutPercentages) {
        val value = (col[row] as Double).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        if (bestValues[row.index()] == value) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}"
        } else {
            "\\cellcolor{cyan!$saturation} $value"
        }
    } else {
        val value = (col[row] as Double).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        val percentage = getPercentage(row, value)
        if (bestValues[row.index()] == value) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}$percentage"
        } else {
            "\\cellcolor{cyan!$saturation} $value$percentage"
        }
    }
}
stringdf

name,0.10,0.20,0.30,0.40,0.50,0.60,0.70
eil101,\cellcolor{cyan!66} 58{\tiny-1.7\%},\cellcolor{cyan!83} 59{\tiny+0.0\%},\cellcolor{cyan!83} 59{\tiny+0.0\%},\cellcolor{cyan!100} \textbf{60*}{\ti...,\cellcolor{cyan!83} 59,\cellcolor{cyan!83} 59{\tiny+0.0\%},\cellcolor{cyan!100} \textbf{60*}{\ti...
gil262,\cellcolor{cyan!42} 132{\tiny-1.5\%},\cellcolor{cyan!71} 136{\tiny+1.5\%},\cellcolor{cyan!100} \textbf{140*}{\t...,\cellcolor{cyan!71} 136{\tiny+1.5\%},\cellcolor{cyan!57} 134,\cellcolor{cyan!64} 135{\tiny+0.7\%},\cellcolor{cyan!92} 139{\tiny+3.7\%}
pr299,\cellcolor{cyan!80} 150{\tiny+0.7\%},\cellcolor{cyan!60} 147{\tiny-1.3\%},\cellcolor{cyan!67} 148{\tiny-0.7\%},\cellcolor{cyan!0} 130{\tiny-12.8\%},\cellcolor{cyan!73} 149,\cellcolor{cyan!93} 152{\tiny+2.0\%},\cellcolor{cyan!100} \textbf{153*}{\t...
lin318,\cellcolor{cyan!67} 179{\tiny+1.1\%},\cellcolor{cyan!83} 182{\tiny+2.8\%},\cellcolor{cyan!100} \textbf{185*}{\t...,\cellcolor{cyan!83} 182{\tiny+2.8\%},\cellcolor{cyan!56} 177,\cellcolor{cyan!72} 180{\tiny+1.7\%},\cellcolor{cyan!100} \textbf{185*}{\t...
rd400,\cellcolor{cyan!60} 192{\tiny-2.0\%},\cellcolor{cyan!100} \textbf{200*}{\t...,\cellcolor{cyan!90} 198{\tiny+1.0\%},\cellcolor{cyan!75} 195{\tiny-0.5\%},\cellcolor{cyan!80} 196,\cellcolor{cyan!75} 195{\tiny-0.5\%},\cellcolor{cyan!0} 174{\tiny-11.2\%}
d493,\cellcolor{cyan!100} \textbf{303*}{\t...,\cellcolor{cyan!96} 302{\tiny+10.2\%},\cellcolor{cyan!24} 280{\tiny+2.2\%},\cellcolor{cyan!10} 276{\tiny+0.7\%},\cellcolor{cyan!4} 274,\cellcolor{cyan!50} 288{\tiny+5.1\%},\cellcolor{cyan!70} 294{\tiny+7.3\%}
u574,\cellcolor{cyan!84} 310{\tiny-1.6\%},\cellcolor{cyan!77} 308{\tiny-2.2\%},\cellcolor{cyan!80} 309{\tiny-1.9\%},\cellcolor{cyan!74} 307{\tiny-2.5\%},\cellcolor{cyan!100} \textbf{315*},\cellcolor{cyan!68} 305{\tiny-3.2\%},\cellcolor{cyan!71} 306{\tiny-2.9\%}
u724,\cellcolor{cyan!100} \textbf{389*}{\t...,\cellcolor{cyan!97} 388{\tiny+3.7\%},\cellcolor{cyan!76} 380{\tiny+1.6\%},\cellcolor{cyan!84} 383{\tiny+2.4\%},\cellcolor{cyan!61} 374,\cellcolor{cyan!94} 387{\tiny+3.5\%},\cellcolor{cyan!79} 381{\tiny+1.9\%}
pcb1173,\cellcolor{cyan!100} \textbf{578*}{\t...,\cellcolor{cyan!86} 570{\tiny+0.5\%},\cellcolor{cyan!94} 575{\tiny+1.4\%},\cellcolor{cyan!96} 576{\tiny+1.6\%},\cellcolor{cyan!80} 567,\cellcolor{cyan!79} 566{\tiny-0.2\%},\cellcolor{cyan!75} 564{\tiny-0.5\%}
fl1400,\cellcolor{cyan!100} \textbf{983*}{\t...,\cellcolor{cyan!72} 956{\tiny+44.2\%},\cellcolor{cyan!0} 877{\tiny+32.3\%},\cellcolor{cyan!0} 712{\tiny+7.4\%},\cellcolor{cyan!0} 663,\cellcolor{cyan!0} 656{\tiny-1.1\%},\cellcolor{cyan!0} 681{\tiny+2.7\%}


In [29]:
fun getLatexTable(formating: String, amountColumns: String, title: String, header: String, label: String, caption: String, body: String): String {
    return """
    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ $formating  }
                \hline
                \multicolumn{$amountColumns}{|c|}{$title} \\
                \hline
                    $header \\
                \hline
                    $body
                \hline
            \end{tabular}
        \end{adjustbox}
        \caption{$caption}
        \label{$label}
    \end{table}
         """
}

val shortAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "TSPrfb"
    Algorithm.DISTANCE -> "TSPrce"
    Algorithm.SPARSITY -> "TSPrcs"
    Algorithm.OP -> "OP"
}
val mediumAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "cluster removal from back"
    Algorithm.DISTANCE -> "cluster removal based on distance"
    Algorithm.SPARSITY -> "cluster removal based on sparsity"
    Algorithm.OP -> "implicit cluster removal"
}

val amountColumns = df.columns().size.toString()
val formating = "|"+ df.columns().joinToString(separator = "") { "p{${width}cm}|" }
val header = df.columnNames().joinToString(separator = " & ")
val label = "tab:$shortAlgString:${mode.name.lowercase()}:${budgetFactor.value}"
val title = "Parameter search: \$R'\$ with \\textit{$shortAlgString}, ${mode.name.lowercase()}, \$ \\gamma = ${budgetFactor.string}\$."
//Parameter run for $R'$ using cluster removal from back with a budget of $\gamma = 0.5$. The percentage value refers to the mean revenue increase compared to $e^{blr}$. The highest revenue of an instance has 100\% saturation decreasing to 0\% at 70\% of the maximum. $\textbf{*}$ refers to the best mean revenue.
val caption = "Parameter run for \$R'\$ using $mediumAlgString with $\\gamma = ${budgetFactor.string}\$. The percentage value refers to the mean revenue increase compared to \$e^{bl${mode.name.lowercase().first().toString()}}$. The highest revenue of an instance has 100\\% saturation decreasing to 0\\% at ${(100 + gradient * -100).toInt()}\\% of the maximum revenue. The larges revenue value is referenced by \$\\textbf{*}\$."

val body = stringdf.rows().joinToString(separator = " \\\\ \n") { row ->
    row.values().joinToString(separator = " & ") {
        it.toString()
    }
} + " \\\\ \\hline " + footer.joinToString(separator = " & ") {
    it.toString()
} + " \\\\"

getLatexTable(formating, amountColumns, title, header, label, caption, body)


    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ |p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|  }
                \hline
                \multicolumn{8}{|c|}{Parameter search: $R'$ with \textit{OP}, flat, $ \gamma = 1.0$.} \\
                \hline
                    name & 0.10 & 0.20 & 0.30 & 0.40 & 0.50 & 0.60 & 0.70 \\
                \hline
                    eil101 & \cellcolor{cyan!66} 58{\tiny-1.7\%} & \cellcolor{cyan!83} 59{\tiny+0.0\%} & \cellcolor{cyan!83} 59{\tiny+0.0\%} & \cellcolor{cyan!100} \textbf{60*}{\tiny+1.7\%} & \cellcolor{cyan!83} 59 & \cellcolor{cyan!83} 59{\tiny+0.0\%} & \cellcolor{cyan!100} \textbf{60*}{\tiny+1.7\%} \\ 
gil262 & \cellcolor{cyan!42} 132{\tiny-1.5\%} & \cellcolor{cyan!71} 136{\tiny+1.5\%} & \cellcolor{cyan!100} \textbf{140*}{\tiny+4.5\%} & \cellcolor{cyan!71} 136{\tiny+1.5\%} & \cellcolor{cyan!57} 134 & \cellcolor{cyan!64} 135{\tiny+0.7\%} & \cellcolor{cyan!92} 